This notebook builds all heatmaps used in the center column of figure 2

In [ ]:
import altair as alt
import pandas as pd
from datetime import datetime

In [ ]:
sge_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251202_SGEsubset.xlsx')
vampseq_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251204_VAMPseqsubset_wDups.xlsx')

alt.data_transformers.disable_max_rows() #gets rid of max data length problem

In [ ]:
sge_df = sge_ppj_subset.loc[sge_ppj_subset['Gene'] == 'RAD51D']
sge_df = sge_df.dropna(subset = ['aa_pos'])
sge_df['aa_pos'] = sge_df['aa_pos'].astype(int)
sge_df.head()

In [ ]:
order = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', '*']
annotation_data =  pd.DataFrame([
    {'start': 1, 'end': 13, 'label': '', 'color':'#888888'},
    {'start': 13, 'end': 61, 'label': 'NTD', 'color': '#B9DBF4'},
    {'start': 61, 'end': 101, 'label': ' ', 'color': '#888888'},
    {'start': 101, 'end': 107, 'label': '  ', 'color': '#C8DBC8'},
    {'start': 107, 'end': 114, 'label': '    ', 'color': '#FF9A00'},
    {'start': 114, 'end': 203, 'label': 'RecA', 'color': '#C8DBC8'},
    {'start': 203, 'end': 206, 'label': '     ', 'color': '#ffc976'},
    {'start': 206, 'end': 310, 'label': 'RecA', 'color': '#C8DBC8'},
    {'start': 310, 'end': 328, 'label': '      ', 'color': '#888888'},
    # Add more annotations as needed
])

threshold_df = pd.DataFrame({'x': [248, 260]})

protein_length = max(sge_df['aa_pos'].tolist()) - 1

df = sge_df.loc[sge_df['aa_pos'] <= protein_length]
column_length = 2
std_width = column_length * protein_length
#std_width = 600

rect_colors = annotation_data['color'].tolist()
domains = annotation_data['label'].tolist()

# Calculate center positions for text
annotation_data['center'] = (annotation_data['start'] + annotation_data['end']) / 2


# Create domain rectangles
annotation_rect = alt.Chart(annotation_data).mark_rect(height=25, 
                                                       stroke = 'black',
                                                      strokeWidth = 2 ).encode(
    x=alt.X('start:Q',
            axis = None,
            scale=alt.Scale(domain=[0, protein_length])),
    x2='end:Q',
    color=alt.Color('color:N', 
                    scale = None,
                    legend= None),
    tooltip=['label', 'start', 'end']
).properties(
    width= std_width,
    height=20
)

#Domain text labels
annotation_text = alt.Chart(annotation_data).mark_text(
    color='black',
    fontSize=20,
    fontWeight='bold',
    baseline='middle',
    dy = -10 # This helps with vertical centering
).encode(
    x=alt.X('center:Q', 
            scale=alt.Scale(domain=[0, protein_length]),
            axis=None
    ), # Position text in the middle of the 50px height
    text='label:N'
)

annotations = alt.layer(annotation_rect, annotation_text).properties(
    width=std_width,
    height=20
)

bins = len(df)
sge_map = alt.Chart(df).mark_rect().encode(
    x = alt.X('aa_pos:Q',
              title = 'Amino Acid Position',
              bin = alt.Bin(maxbins = bins, minstep = 1),
              axis = alt.Axis(values = list(range(0, bins, 100)),
                              labelFontSize = 22, 
                              titleFontSize = 24
                             ),
              scale = alt.Scale(domain = [0, protein_length])
             ),
    y = alt.Y('aa_alt:O',
              title = '',
              sort = order,
              axis = alt.Axis(
                  labelFontSize = 18, 
                  labelFontWeight = 'bold',
                  titleFontSize = 24
              )
             ),
    color = alt.Color('auth_reported_score',
                      scale = alt.Scale(
                          domain = [-0.2, 0],
                          clamp = True,
                          scheme = 'bluepurple',
                          reverse = True
                      ),
                      legend = alt.Legend(title = 'Score',
                                          labelFontSize = 20,
                                          titleFontSize = 22
                                         ),
                     ),
    tooltip = ['auth_reported_score']
).properties(
    width = std_width,
    height = 300
)


threshl = alt.Chart(threshold_df).mark_rule(color='#5a5a5a', strokeDash=[8,8], strokeWidth=2).encode(
    x=alt.X('x:Q', scale=alt.Scale(domain=[0, protein_length]))
)

sge_map = alt.layer(sge_map, threshl)

sge_map = alt.vconcat(annotations, sge_map, spacing = -5).resolve_scale(
    x = 'shared',
    color = 'independent'
).configure_axis(
    grid = False
).configure_view(
    stroke = None
).properties(
    title = alt.TitleParams(text = 'RAD51D',
                            anchor = 'middle',
                            align = 'center',
                            fontSize = 24
                           )
)

sge_map.display()

In [ ]:
df = sge_ppj_subset.loc[sge_ppj_subset['Gene'] == 'RAD51D']

df.loc[df['consequence'].str.contains('missense'), 'consequence'] = 'Missense'
df.loc[df['consequence'] == 'synonymous_variant', 'consequence'] = 'Synonymous'
df.loc[df['consequence'] == 'intron_variant', 'consequence'] = 'Intron'
df.loc[df['consequence'] == 'stop_gained', 'consequence'] = 'Stop Gained'
df.loc[df['consequence'] == 'stop_lost', 'consequence'] = 'Stop Lost'
df.loc[df['consequence'].str.contains('site'), 'consequence'] = 'Canonical Splice'
df.loc[df['consequence'].str.contains('ing_var'), 'consequence'] = 'Splice Region'
df.loc[df['consequence'].str.contains('UTR'), 'consequence'] = 'UTR Variant'
df.loc[df['consequence'] == 'start_lost', 'consequence'] = 'Start Lost'

In [ ]:
palette = [
    '#006616', # dark green,
    '#81B4C7', # dusty blue
    '#ffcd3a', # yellow
    '#6AA84F', # med green
    '#888888', # med gray
    '#1170AA', # darker blue
    '#CFCFCF' # light gray
        
    ]
    
    
variant_types = [
    'Synonymous',
    'Missense',  
    'Stop Gained',
    'Intron', 
    'Stop Lost',
    'Canonical Splice', 
    'Splice Region',
]

df = df.loc[(df['hg38_start'] <= 35101370) & (df['hg38_start'] >= 35101325)]

ref_df = df.groupby('hg38_start').agg({
    'ref_allele': 'first'
}).reset_index()
ref_df['alt'] = ref_df['ref_allele']

missing_bases = pd.DataFrame({'hg38_start': [35101330, 35101331, 35101332],
                              'ref_allele': ['C', 'C', 'C'],
                             'alt': ['C', 'C', 'C']}
                              )

ref_df = pd.concat([ref_df, missing_bases])

end = df['hg38_start'].max()
start = df['hg38_start'].min()

exon_domain = [start, end]

bins = (end - start) + 1

rect_size = 15
spacing = 7.5

total_width = (rect_size + spacing) * bins - spacing
total_height = (rect_size + spacing) * 4 - spacing



map = alt.Chart(df).mark_square(strokeWidth = 3).encode(
    x = alt.X('hg38_start:O', 
              title = '',
              axis = alt.Axis(values = list(range(start, end + 1)),
                              labels = False, 
                              ticks = False
                             ),
              scale = alt.Scale(domain = list(range(start, end + 1)),
                                reverse = True
                               )
             ),
    y = alt.Y('alt_allele:N',
             title = '',
             axis = alt.Axis(labelFontSize = 18)
             ),
    size = alt.value(rect_size * rect_size),
    color = alt.Color('auth_reported_score:Q',
                      scale = alt.Scale(
                          scheme = 'bluepurple',
                          domain = [-0.2, 0],
                          clamp = True,
                          reverse = True
                      ),
                      legend = None
                     ),
    stroke = alt.Stroke('consequence',
                        scale = alt.Scale(
                            range = palette,
                            domain = variant_types
                        ),
                        legend = alt.Legend(
                            symbolFillColor = 'white',
                            title = 'Consequence',
                            titleFontSize = 20,
                            labelFontSize = 18,
                            columns = 7,
                            orient = 'bottom',
                            titleAnchor = 'start',
                            titleOrient = 'left',
                            symbolStrokeWidth = 3
                        )
                       )
).properties(
    width = total_width,
    height = total_height
)

text = alt.Chart(ref_df).mark_text(fontSize = 20).encode(
    x = 'hg38_start:O',
    y = 'alt',
    text = 'ref_allele'
)

map = alt.layer(map, text).configure_view(
    stroke = 'black'
).configure_axis(
    domainColor = 'black',
    tickColor = 'black'
)

map.display()

In [ ]:
vampseq_df = vampseq_ppj_subset.loc[vampseq_ppj_subset['Gene'] == 'G6PD']
vampseq_df  = vampseq_df.drop_duplicates(subset = ['ID'])
vampseq_df = vampseq_df.dropna(subset = ['aa_pos'])
vampseq_df['aa_pos'] = vampseq_df['aa_pos'].astype(int)

vampseq_df.head()

In [ ]:
protein_length = max(df['aa_pos'].tolist()) - 1

column_length = 3
std_width = column_length * protein_length
order = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', '*']

annotation_data = pd.DataFrame([
    {'start': 35, 'end': 210, 'label': 'NAD Binding Domain', 'color': '#B9DBF4'},
    {'start': 212, 'end': 503, 'label': 'C-Terminal Domain', 'color': '#C8DBC8'},
    {'start': 1, 'end': 35, 'label': '', 'color': 'grey'},
    {'start': 210, 'end': 212, 'label': '', 'color': 'grey'},
    {'start': 503, 'end': 515, 'label': '', 'color': 'grey'}
    # Add more annotations as needed
])

rect_colors = ['#B9DBF4','#C8DBC8','grey', 'grey', 'grey']
domains = ['NAD Binding Domain', 'C-Terminal Domain', '', '', '']

# Calculate center positions for text
annotation_data['center'] = (annotation_data['start'] + annotation_data['end']) / 2


# Create domain rectangles
annotation_rect = alt.Chart(annotation_data).mark_rect(height=25, 
                                                       stroke = 'black',
                                                      strokeWidth = 2 ).encode(
    x=alt.X('start:Q',
            axis = None,
            scale=alt.Scale(domain=[0, 515])),
    x2='end:Q',
    color=alt.Color('label:N', 
                    scale = alt.Scale(domain = domains,
                                      range = rect_colors
                                     ),
                    legend= None),
    tooltip=['label', 'start', 'end']
).properties(
    width=std_width,
    height=20
)

#Domain text labels
annotation_text = alt.Chart(annotation_data).mark_text(
    color='black',
    fontSize=20,
    fontWeight='bold',
    baseline='middle',
    dy = -10 # This helps with vertical centering
).encode(
    x=alt.X('center:Q', 
            scale=alt.Scale(domain=[0,515]),
            axis=None
    ), # Position text in the middle of the 50px height
    text='label:N'
)

annotations = alt.layer(annotation_rect, annotation_text).properties(
    width=std_width,
    height=20
)

map = alt.Chart(vampseq_df).mark_rect().encode(
    x = alt.X('aa_pos:Q',
              title = 'Amino Acid Position',
              bin = alt.Bin(maxbins = protein_length + 1),
              axis = alt.Axis(values = list(range(0, 530, 50)),
                             labelFontSize = 22,
                             titleFontSize = 24)
             ),
    y = alt.Y('aa_alt',
              title = '',
              sort = order,
              axis = alt.Axis(labelFontSize = 18,
                              labelFontWeight = 'bold'
                             )
             ),
    color = alt.Color('auth_reported_score:Q',
                      title = 'Score',
                      scale = alt.Scale(
                          scheme = 'bluepurple',
                          domain = [0, 1],
                          clamp = True,
                          reverse = True
                      ),
                      legend = alt.Legend(
                          titleFontSize = 24,
                          labelFontSize = 22
                      )
                     )
).properties(
    width = std_width,
    height = 325
)

map = alt.vconcat(annotations, map, spacing = -5).configure_view(
    stroke = None
).properties(
    title = alt.TitleParams('G6PD',
                            anchor = 'middle', 
                            align = 'center',
                            fontSize = 24
                           )
)

map.display()